# 01 — Data Understanding (RAND HRS 1992–2022 v1)

This notebook:
1) checks data path
2) loads a safe sample (first 100k rows) from the .dta file
3) produces variable overview (dtype, missingness)
4) searches key variables by keywords
5) saves outputs for later steps

In [ ]:
import pandas as pd
from pathlib import Path

# Project root = parent of notebooks/
PROJECT_ROOT = Path().resolve().parents[0]
DATA_PATH = PROJECT_ROOT / "data_raw" / "randhrs1992_2022v1.dta"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH)
print("exists:", DATA_PATH.exists())

In [ ]:
# Read first chunk (safe sample)
CHUNK_SIZE = 100_000

it = pd.read_stata(
    str(DATA_PATH),
    convert_categoricals=False,  # keep as numeric codes, faster/less memory
    chunksize=CHUNK_SIZE
)

df = next(it)  # first 100k rows only
print("sample shape:", df.shape)
df.head(3)

In [ ]:
df.info()

print("\nMissing rate (overall):", round(df.isnull().mean().mean() * 100, 2), "%")
print("Approx memory usage (MB):", round(df.memory_usage(deep=True).sum() / 1024**2, 2))

In [ ]:
var_overview = pd.DataFrame({
    "variable": df.columns,
    "dtype": [str(t) for t in df.dtypes],
    "non_null": df.notnull().sum().values,
    "missing_pct": (df.isnull().mean().values * 100).round(2),
})

var_overview.sort_values("missing_pct").head(30)

In [ ]:
out_dir = PROJECT_ROOT / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

var_path = out_dir / "variable_overview_sample_100k.csv"
var_overview.to_csv(var_path, index=False)

print("saved:", var_path)

In [ ]:
keywords = [
    "age", "sex", "male", "female", "gender",
    "educ", "education", "race", "hisp", "mar", "marital",
    "adl", "iadl",
    "lonely", "isolation", "social", "friend", "family",
    "cesd", "depress", "depression",
    "smoke", "bmi",
    "diab", "diabetes", "hyper", "hypertension",
    "heart", "stroke", "cancer"
]

hits = [c for c in df.columns if any(k in c.lower() for k in keywords)]
print("hit count:", len(hits))
hits[:80]

In [ ]:
hits_path = out_dir / "keyword_hits_sample_100k.txt"
hits_path.write_text("\n".join(hits), encoding="utf-8")

print("saved:", hits_path)

In [ ]:
# quick heuristic: fewer missing + keyword matched
hit_df = var_overview[var_overview["variable"].isin(hits)].copy()
hit_df = hit_df.sort_values(["missing_pct", "variable"])

hit_df.head(30)

In [ ]:
sample_path = out_dir / "sample_100k.parquet"
df.to_parquet(sample_path, index=False)

print("saved:", sample_path)